# 🔄 ONNX Export & Validation

**Objective**: Convert PyTorch YOLO11 model to ONNX for production deployment.

**Environment**: Local Windows machine (CPU-only is fine).

## 📋 Setup

In [ ]:
# Install dependencies
!pip install ultralytics onnx onnxruntime -q

from pathlib import Path
from ultralytics import YOLO
import onnx
import onnxruntime as ort
import numpy as np
from PIL import Image

print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

## 🔄 Export to ONNX

In [ ]:
# Load trained PyTorch model
model_path = Path(r'c:\Users\georgem\source\repos\AutoFactoryScope\models\best.pt')
model = YOLO(model_path)

# Export to ONNX
onnx_path = model.export(
    format='onnx',
    imgsz=512,  # Must match training imgsz
    dynamic=False,  # Fixed input size for faster inference
    simplify=True,  # Simplify ONNX graph (faster)
    opset=12,  # ONNX opset version (12 is stable)
)

print(f"\nONNX model exported to: {onnx_path}")

In [ ]:
# Rename to production name
final_path = Path(r'c:\Users\georgem\source\repos\AutoFactoryScope\models\robot_detector.onnx')
Path(onnx_path).rename(final_path)

print(f"Final ONNX model: {final_path}")
print(f"File size: {final_path.stat().st_size / 1024 / 1024:.2f} MB")

## ✅ Validate ONNX Model

In [ ]:
# Load ONNX model
onnx_model = onnx.load(final_path)

# Check validity
onnx.checker.check_model(onnx_model)
print("✓ ONNX model is valid")

# Inspect inputs/outputs
print("\nModel Inputs:")
for inp in onnx_model.graph.input:
    print(f"  {inp.name}: {[d.dim_value for d in inp.type.tensor_type.shape.dim]}")

print("\nModel Outputs:")
for out in onnx_model.graph.output:
    print(f"  {out.name}: {[d.dim_value for d in out.type.tensor_type.shape.dim]}")

## 🧪 Test ONNX Inference

In [ ]:
# Create ONNX Runtime session
session = ort.InferenceSession(str(final_path), providers=['CPUExecutionProvider'])

# Get input/output names
input_name = session.get_inputs()[0].name
output_names = [out.name for out in session.get_outputs()]

print(f"Input name: {input_name}")
print(f"Output names: {output_names}")

In [ ]:
# Preprocess test image
def preprocess_image(image_path: Path, input_size: int = 512):
    """Preprocess image for ONNX inference."""
    img = Image.open(image_path).convert('RGB')
    img = img.resize((input_size, input_size))
    img_array = np.array(img).astype(np.float32) / 255.0
    img_array = np.transpose(img_array, (2, 0, 1))  # HWC -> CHW
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dim
    return img_array

# Test on a sample image from your dataset
test_image_path = Path(r'c:\Users\georgem\source\repos\AutoFactoryScope\datasets\v2_roboflow_export\test\images')
test_images = list(test_image_path.glob('*.jpg'))[:1]

if test_images:
    img_tensor = preprocess_image(test_images[0])
    print(f"Input tensor shape: {img_tensor.shape}")
    
    # Run inference
    outputs = session.run(output_names, {input_name: img_tensor})
    
    print(f"Output shape: {outputs[0].shape}")
    print(f"\n✓ ONNX inference successful!")
else:
    print("⚠ No test images found. Update test_image_path.")

## 📊 Compare PyTorch vs ONNX (Parity Check)

In [ ]:
# PyTorch inference
pt_results = model.predict(
    source=str(test_images[0]),
    imgsz=512,
    conf=0.25,
    verbose=False
)

# ONNX inference (already ran above)
onnx_outputs = outputs[0]

# Parse ONNX outputs
# YOLO output format: [x_center, y_center, w, h, class_scores...]
onnx_output = onnx_outputs[0].T  # Transpose to (num_preds, num_classes+4)
onnx_boxes = onnx_output[:, :4]
onnx_scores = onnx_output[:, 4:].max(axis=1)
onnx_detections = (onnx_scores > 0.25).sum()

# PyTorch detections
pt_boxes = pt_results[0].boxes
pt_detections = len(pt_boxes)

print(f"\nParity Check:")
print(f"  PyTorch detections: {pt_detections}")
print(f"  ONNX detections: {onnx_detections}")
print(f"  Difference: {abs(pt_detections - onnx_detections)}")

if abs(pt_detections - onnx_detections) <= 2:
    print("\n✅ ONNX parity confirmed (< 2 detection difference)")
else:
    print("\n⚠ Large difference detected. Review preprocessing.")

## 🎯 Benchmark Inference Speed

In [ ]:
import time

# Warmup
for _ in range(10):
    session.run(output_names, {input_name: img_tensor})

# Benchmark
num_runs = 100
start = time.time()
for _ in range(num_runs):
    session.run(output_names, {input_name: img_tensor})
end = time.time()

avg_time_ms = (end - start) / num_runs * 1000
fps = 1000 / avg_time_ms

print(f"\nBenchmark Results (CPU):")
print(f"  Average inference time: {avg_time_ms:.2f} ms")
print(f"  Throughput: {fps:.2f} FPS")
print(f"\nNote: Production will use tiling, so multiply by number of tiles.")

## 📝 Create Model Card

In [ ]:
model_card = f"""# Model Card: Robot Detector v1

## Model Details

- **Model Type**: YOLO11n (Nano)
- **Task**: Object Detection (Factory Layout Robots)
- **Framework**: Ultralytics YOLOv11
- **License**: AGPL-3.0 (internal use only)
- **Training Date**: {Path(model_path).stat().st_mtime}

## Performance

- **mAP50**: [Fill from training results]
- **mAP50-95**: [Fill from training results]
- **Inference Speed**: {avg_time_ms:.2f} ms (CPU, single tile)
- **Model Size**: {final_path.stat().st_size / 1024 / 1024:.2f} MB (ONNX)

## Training Data

- **Dataset**: v2_roboflow_export
- **Images**: 138 (train + val + test)
- **Classes**: robot (1 class)
- **Augmentation**: Mosaic, flip, HSV, rotation

## Usage

```python
import onnxruntime as ort

session = ort.InferenceSession('models/robot_detector.onnx')
# ... (see inference.py for full example)
```

## Limitations

- Trained on 2D layout drawings only
- Single class (robot) - no equipment types yet
- May struggle with very small symbols (<20px)

## Next Steps

- [ ] Add more classes (weld_gun, fixture, conveyor, gate)
- [ ] Active learning iteration with production data
- [ ] INT8 quantization for faster inference
"""

model_card_path = Path(r'c:\Users\georgem\source\repos\AutoFactoryScope\models\MODEL_CARD.md')
model_card_path.write_text(model_card)

print(f"Model card saved to: {model_card_path}")

## ✅ Done!

Your ONNX model is ready for production deployment.

**Next Steps:**

1. Update backend config to use new model path
2. Test tiled inference with large factory layouts
3. Run full integration test with FastAPI